# Track 07 — 안전 · HITL · 관측 (Safety, HITL & Observability)

### 안전·HITL·관측이란?

에이전트가 도구를 직접 호출하기 시작하면 "틀린 답"보다 "되돌릴 수 없는 행동"이 더 큰 리스크가 됩니다. 이 트랙은 그 리스크를 세 겹으로 줄입니다 — **사람 승인(HITL)** 으로 위험한 행동을 사람이 막고, **sanitize** 로 검색 결과·도구 출력·로그에 숨은 프롬프트 인젝션과 비밀을 무력화하고, **관측 트레이스** 로 무슨 일이 있었는지 표준 필드에 남깁니다. trace 가 없으면 디버깅도 평가도 할 수 없습니다.

### 이 노트북에서 보여줄 것

| Session | 무엇을 | 왜 |
|---|---|---|
| **1. HITL** | `safe_lookup`(읽기 전용)과 `sensitive_publish`(승인 필수) 두 도구로 승인·거부·읽기전용 시나리오 3건을 mock stdin 으로 검증 | 승인 게이트를 도구 `execute()` *안* 에 두면 LLM 이 무엇을 호출하든 사람이 최종적으로 막을 수 있음을 봅니다. |
| **2. Prompt Injection** | injection 10종(user·rag_chunk·tool·log)을 두 sanitizer 에 통과시키고, 구조화 JSON 경로의 NFC 정규화까지 | 구조 태그 무력화·비밀 마스킹이 *무엇을 막고 무엇을 남기는지*(자연어 지시는 데이터로 남음)를 구분합니다. |
| **3. Observability** | 표준 필드 상수로 짧은 `ToolAgent` 실행을 trace 이벤트(JSONL)와 필드 glossary 로 남김 | 모든 팀이 같은 키로 로그를 남겨야 트레이스를 집계·평가할 수 있음을 봅니다. |

### 이 노트북을 마치면

- 위험 도구 앞에 HITL 승인 게이트를 붙이고, 거부를 `error="rejected_by_human"` 로 판정할 수 있습니다.
- RAG·도구·로그에 섞인 인젝션 페이로드와 비밀을 sanitize·마스킹할 수 있습니다.
- `ToolAgent` 실행을 표준 관측 필드로 JSONL 트레이스에 남기고 팀용 필드 사전을 만들 수 있습니다.

**요약:** Session 1-2·2·3-1·3-3 은 키 없이 오프라인으로 실행되고, 라이브 LLM 셀(Session 1-3·3-2)만 `EXAONE_API_KEY` 가 필요합니다(없으면 `[SKIP]`).

**산출물:** `_out/hitl_trace.json`, `_out/injection_defense.json`, `_out/session_trace.jsonl`, `_out/field_glossary.md`

In [ ]:
import json
import os
import sys
import time
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

# (en) Requires editable install at repo root: pip install -r requirements.txt && pip install -e ./exaone
# (kr) 저장소 루트에서 editable 설치 필요: pip install -r requirements.txt && pip install -e ./exaone
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "exaone 가 설치되지 않았습니다. 저장소 루트에서 "
        "pip install -r requirements.txt && pip install -e ./exaone 후 커널을 재시작하세요."
    ) from exc

exaone.load_project_env()
# (en) Gate live LLM steps on API key presence (offline sessions run without a key).
# (kr) API 키 유무로 라이브 LLM 단계를 가드한다(오프라인 세션은 키 없이 실행).
HAS_API = bool(os.environ.get("EXAONE_API_KEY", "").strip())
ROOT = exaone.project_root()
TRACK07 = ROOT / "recipes" / "track07_safety_hitl_observability"
DATA = TRACK07 / "data"
out_dir = Path("_out")
out_dir.mkdir(parents=True, exist_ok=True)

model = os.environ.get("EXAONE_MODEL", "").strip() or exaone.llm.ExaoneClient.DEFAULT_MODEL
# (en) Only build the client when a key exists; otherwise leave it None so offline cells survive.
# (kr) 키가 있을 때만 클라이언트 생성; 없으면 None 으로 둬 오프라인 셀이 살아남게 한다.
client = exaone.integrations.build_llm_from_env() if HAS_API else None

print("exaone", exaone.__version__, "| model:", model, "| HAS_API =", HAS_API)

**출력 해석:** `exaone 0.1.0 | model: LGAI-EXAONE/K-EXAONE-236B-A23B | HAS_API = True` 한 줄이 보이면 준비 완료입니다.

- `HAS_API = True` 면 `EXAONE_API_KEY` 가 있어 라이브 LLM 셀(Session 1-3·3-2)이 실제로 호출됩니다. `False` 면 `client = None` 이라 그 두 셀만 `[SKIP]` 되고 나머지 오프라인 셀은 정상 실행됩니다.
- `client`·`DATA` 가 이 노트북에서 쓰는 LLM 클라이언트와 데이터 경로입니다(`model` 은 배너 표시용).

## Session 1. Human-in-the-Loop

**테스트 시나리오** — `hitl_scenarios` (`hitl_scenarios.json`) 3건. 승인·거부·읽기전용을 mock stdin 으로 검증합니다.

| id | 기대 도구 | stdin |
|---|---|---|
| `lookup_only` | safe_lookup | (승인 불필요) |
| `publish_approved` | sensitive_publish | yes |
| `publish_rejected` | sensitive_publish | n |

도구를 두 종류로 나눕니다:


### Session 1-1. `HumanApprovalController` + 도구 정의

**하는 일:** `HumanApprovalController` + 도구 정의.

**정상:** `HITL registry builder ready` 가 보임

**의미:** 도구를 두 종류로 나눕니다: 읽기 전용(safe_lookup, 승인 불필요)과 부수효과 있는(sensitive_publish, 사람 승인 필수). 승인 게이트는 도구 실행 *안* 에 있어, LLM 이 무엇을 호출하든 사람이 막을 수 있습니다.


In [ ]:
MOCK_KB = {
"refund": "Refunds within 14 days with receipt.",
"shipping": "Shipping 3-5 business days.",
"privacy": "We do not sell personal data.",
}


@dataclass
class HumanApprovalController:
    # (en) Terminal y/N approval gate for dangerous tools.
    # (kr) 위험 도구용 터미널 y/N 승인 게이트.
    auto_approve: bool
    _stdin: Callable[[], str] | None = None

    def request_approval(self, tool_name: str, args: dict[str, Any]) -> bool:
        if self.auto_approve:
            return True
        line = (self._stdin() if self._stdin else sys.stdin.readline()) or ""
        return line.strip().lower() in ("y", "yes")


def exec_safe_lookup(_name: str, args: dict[str, Any]) -> dict[str, Any]:
    topic = (args.get("topic") or "").strip().lower()
    if not topic:
        return exaone.tools.ToolResult.validation_error(source="safe_lookup", error="topic required").to_dict()
    for key, text in MOCK_KB.items():
        if key in topic:
            return exaone.tools.ToolResult.success(content=text, source="safe_lookup", metadata={"matched": key}).to_dict()
    return exaone.tools.ToolResult.empty(source="safe_lookup", reason="no match").to_dict()


# (en) Dangerous tool: the human approval gate lives inside execute().
# (kr) 위험 도구: 사람 승인 게이트가 execute() 안에 있다.
def make_sensitive_publish(approval: HumanApprovalController):
    def exec_sensitive_publish(_name: str, args: dict[str, Any]) -> dict[str, Any]:
        title = (args.get("title") or "").strip()
        body = (args.get("body") or "").strip()
        if not title or not body:
            return exaone.tools.ToolResult.validation_error(source="sensitive_publish", error="title and body required").to_dict()
        if not approval.request_approval("sensitive_publish", {"title": title, "body_preview": body[:120]}):
            return exaone.tools.ToolResult.failure(source="sensitive_publish", error="rejected_by_human").to_dict()
        pub_id = str(uuid.uuid4())[:8]
        return exaone.tools.ToolResult.success(content=f"published id={pub_id}", source="sensitive_publish", metadata={"publish_id": pub_id}).to_dict()

    return exec_sensitive_publish


SAFE_LOOKUP_SCHEMA = {
"type": "function",
"function": {
"name": "safe_lookup", "description": "Read-only policy lookup. No human approval.",
"parameters": {"type": "object", "required": ["topic"], "properties": {"topic": {"type": "string"}}, "additionalProperties": False},
},
}
SENSITIVE_PUBLISH_SCHEMA = {
"type": "function",
"function": {
"name": "sensitive_publish", "description": "External publish — requires human approval in this notebook.",
"parameters": {"type": "object", "required": ["title", "body"],
               "properties": {"title": {"type": "string"}, "body": {"type": "string"}}, "additionalProperties": False},
},
}


def build_registry(approval: HumanApprovalController) -> exaone.tools.ToolRegistry:
    reg = exaone.tools.ToolRegistry()
    reg.register(exaone.tools.tool_from_callable("safe_lookup", SAFE_LOOKUP_SCHEMA, exec_safe_lookup))
    reg.register(exaone.tools.tool_from_callable("sensitive_publish", SENSITIVE_PUBLISH_SCHEMA, make_sensitive_publish(approval)))
    return reg


print("HITL registry builder ready")


**출력 해석:** `HITL registry builder ready` 가 보이면 두 종류의 도구가 등록된 것입니다.

- `safe_lookup` 은 읽기 전용이라 승인 없이 실행됩니다.
- `sensitive_publish` 는 외부 게시처럼 부수효과가 있어 `execute()` *안* 에 사람 승인 게이트가 박혀 있습니다. 승인 로직이 도구 내부에 있으므로 LLM 이 무엇을 호출하든 사람이 최종적으로 막을 수 있습니다.

### Session 1-2. hitl_scenarios 실행

**하는 일:** `hitl_scenarios` 3건을 mock stdin 으로 돌립니다 (LLM 없이).

**입력:** `data/hitl_scenarios.json`

**정상:** `시나리오:` 3줄 + 시나리오별 pass

**의미:** 사람 승인 게이트가 거부 시 `rejected_by_human` 으로 끝나는지 봅니다.


In [ ]:
scenarios = json.loads((DATA / "hitl_scenarios.json").read_text(encoding="utf-8"))
print("시나리오: HITL", len(scenarios), "건 (hitl_scenarios.json)")
for s in scenarios:
    print(f"  {s['id']}: expect={s.get('expect_tool')}, stdin={s.get('stdin_response', '—')}")
offline_results = []

for sc in scenarios:
    if sc["id"] == "lookup_only":
        reg = build_registry(HumanApprovalController(auto_approve=True))
        out = reg.execute("safe_lookup", {"topic": "refund"})
        offline_results.append({"id": sc["id"], "tool": "safe_lookup", "ok": out.get("ok"), "outcome": out.get("outcome")})
        continue
    stdin_line = sc.get("stdin_response", "yes")
    reg = build_registry(HumanApprovalController(auto_approve=False, _stdin=lambda line=stdin_line: line))
    out = reg.execute("sensitive_publish", {"title": "Test title", "body": "Test body for HITL gate."})
    offline_results.append({"id": sc["id"], "tool": "sensitive_publish", "stdin": stdin_line, "ok": out.get("ok"), "outcome": out.get("outcome"), "error": out.get("error")})

print(json.dumps(offline_results, ensure_ascii=False, indent=2))
# (en) The rejected scenario must fail with a human-rejection error.
# (kr) 거부 시나리오는 사람 거부 에러로 실패해야 한다.
assert offline_results[2]["error"] == "rejected_by_human" 

**출력 해석:** 세 시나리오가 mock stdin 으로 LLM 없이 검증됩니다.

- `lookup_only` → `ok: true`, `outcome: success` (읽기 전용, 승인 불필요).
- `publish_approved` (stdin `yes`) → `ok: true`, `outcome: success` — 사람이 승인해 게시됩니다.
- `publish_rejected` (stdin `n`) → `ok: false`, `error: "rejected_by_human"`. 거부 판정은 **`error` 필드로** 읽습니다 — `ToolResult.failure()` 는 `outcome` 을 일반 실패 버킷 `transport_error` 로 남기므로, `outcome` 만 보면 네트워크 오류로 오해할 수 있습니다(assert 도 `error` 를 확인합니다).

### Session 1-3. (라이브) `ToolAgent` + LLM — auto-approve 데모

**하는 일:** 키가 있으면 실제 `ToolAgent` 를 auto-approve 로 실행하고, 없으면 `[SKIP]` 합니다.

**정상:** `success: True` 와 한국어 구조화 `answer_preview` (키 없으면 `[SKIP]`)

**의미:** LLM 이 `safe_lookup` 을 호출해 근거를 모으고 높임말 답을 만드는 정상(통과) 경로를 봅니다 — 거부 게이트가 아닌 흐름입니다.

In [ ]:
live_trace = None
if HAS_API:
    reg = build_registry(HumanApprovalController(auto_approve=True))
    agent = exaone.agents.ToolAgent(
        tool_registry=reg,
        system_prompt="Policy assistant. Use safe_lookup for refund/shipping/privacy. Call sensitive_publish only when user explicitly asks to publish.",
        use_thinking_router=False, max_turns=6,
    )
    query = "환불은 며칠 이내 가능해? (refund policy)"
    t0 = time.monotonic()
    res = agent.run(exaone.agents.AgentContext(query=query), llm=client)
    live_trace = {
        "query": query, "success": res.success,
        "latency_ms": round((time.monotonic() - t0) * 1000, 1),
        "answer_preview": (res.content or "")[:200], "metadata_keys": list((res.metadata or {}).keys()),
    }
    print(live_trace)
else:
    # (en) No key: skip the live agent demo; the offline scenarios above already pass.
    # (kr) 키 없음: 라이브 데모 생략(위 오프라인 시나리오는 이미 통과).
    print("[SKIP] LLM 키 없음 — 라이브 ToolAgent 데모 건너뜀")

**출력 해석:** 키가 있으면 실제 `ToolAgent` 가 auto-approve 로 돌며 정상(통과) 경로를 보여줍니다(키 없으면 `[SKIP]`).

- `success: True`, `sources: ["safe_lookup"]` — 에이전트가 `safe_lookup` 결과를 근거로 답했습니다. `answer` 본문은 라이브 LLM 이라 run 마다 문구가 조금씩 달라지지만(예: "환불은 영수증과 함께 14일 이내에 가능합니다."), **높임말**과 근거 인용은 일관됩니다. `confidence` 는 run 에 따라 수치(`0.95`)나 라벨(`"high"`)로 옵니다.
- `latency_ms` 와 `metadata_keys`(`turns_used`·`tool_invocations`·`llm_calls` 등)는 이어지는 Session 3 의 관측 트레이스로 연결됩니다.

### Session 1-4. 산출물 — `hitl_trace.json`

**하는 일:** 오프라인 3건 + (키가 있으면) 라이브 결과를 `hitl_trace.json` 으로 저장합니다.

**정상:** `saved:` 경로가 출력됨

**의미:** 이 파일은 다음 Session 이나 회귀 테스트의 입력이 됩니다.

In [ ]:
payload = {
"generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
"offline_scenarios": offline_results,
"live_tool_agent": live_trace,
}
path = out_dir / "hitl_trace.json"
path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", path.resolve())


**출력 해석:** `saved:` 경로가 보이면 `hitl_trace.json` 이 `_out/` 에 기록된 것입니다.

- 오프라인 3건 + (키가 있으면) 라이브 1건이 한 파일에 모입니다.
- 이 산출물은 거부 시나리오가 `rejected_by_human` 으로 남았는지 확인하는 고정 증거로, 회귀 테스트의 입력이 됩니다.

## Session 2. Prompt Injection & Untrusted Text

**테스트 시나리오** — `injection_cases` (`injection_cases.jsonl`) 10건. user·rag_chunk·tool·log 벡터별 sanitizer 방어를 봅니다.

검색 청크·사용자 입력·로그에 섞인 악성 지시를 두 sanitizer 로 막습니다.

### Session 2-1. injection_cases 10건

**하는 일:** `injection_cases` 10건을 sanitizer 에 통과시킵니다.

**입력:** `data/injection_cases.jsonl`

**정상:** `시나리오:` + `pass count: 10 / 10` + 대표 케이스 sanitize preview

**의미:** 케이스 id·벡터(user/rag_chunk/tool/log)별로 방어가 되는지 봅니다.

In [ ]:
cases = [json.loads(line) for line in (DATA / "injection_cases.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
print("시나리오: injection 방어", len(cases), "건 (injection_cases.jsonl)")
for c in cases[:3]:
    print(f"  {c['id']} ({c.get('vector', '?')}):", (c.get('text') or c.get('injection_text') or '')[:45] + "…")


# (en) Same defenses as Track 04 §5: delimiter tags, Question: prefix, log redaction, benign rows.
# (kr) Track 04 §5 와 동일: 구분자 태그, Question: 접두, 로그 마스킹, 정상 행.
def defense_pass(case, raw, untrusted, sanitized):
    cid = case["id"]
    if case["vector"] == "log":
        return "[REDACTED]" in sanitized and "secret-token" not in sanitized
    if cid in ("inj07", "inj08"):
        return True
    if "[removed-tag:" in untrusted:
        return True
    if "Question:" in raw and "[ref]" in untrusted:
        return True
    if "Bearer" in raw and "[REDACTED]" in sanitized:
        return True
    return False


rows = []
for case in cases:
    raw = case["text"]
    if case["vector"] == "log":
        sanitized = exaone.observability.sanitize_for_log(raw)
        untrusted = raw
    else:
        untrusted = exaone.context_management.sanitize_untrusted_reference_text(raw)
        sanitized = exaone.observability.sanitize_for_log(untrusted)
    ok = defense_pass(case, raw, untrusted, sanitized)
    rows.append({"id": case["id"], "vector": case["vector"], "pass": ok, "changed": sanitized != raw or untrusted != raw, "preview": sanitized[:160]})

passed = sum(1 for r in rows if r["pass"])
print("pass count:", passed, "/", len(rows))
# (en) Show representative sanitized previews so the defense is visible in this very cell.
# (kr) 대표 케이스의 sanitize 결과를 이 셀에서 바로 보여 방어를 눈으로 확인.
for cid in ("inj01", "inj06", "inj10"):
    row = next(r for r in rows if r["id"] == cid)
    print(f'  {row["id"]} ({row["vector"]}): {row["preview"][:80]}')
assert passed == len(rows), [r for r in rows if not r["pass"]]

**출력 해석:** injection 10건이 모두 `pass` 이고, 위에 출력된 대표 preview(inj01·inj06·inj10)로 sanitizer 가 *무엇을* 막는지 직접 보입니다.

- **구조 무력화:** 가짜 닫는 태그가 `[removed-tag:...]` 로 바뀝니다(inj01·02·03·04·09). 비신뢰 텍스트가 reference 구획을 깨지 못하게 하는 것이지 지시 문장을 삭제하는 것은 아닙니다 — inj01 의 `Ignore previous instructions…` 는 구조가 무력화된 평문(데이터)으로 남습니다(의도된 동작). `Question:` 으로 시작하는 inj02·05 에는 `[ref]` 접두가 함께 붙습니다.
- **비밀 마스킹:** `Bearer …` 토큰과 로그의 `authorization` 값이 `[REDACTED]` 로 가려집니다(inj06·inj10).
- **정상 통과:** inj07·inj08 같은 benign 입력은 변형 없이(`changed: false`) 통과합니다.

### Session 2-2. Unicode 정규화 (구조화 JSON 경로)

**하는 일:** 구조화 출력의 문자열 값에 `normalize_json_string_values` 를 적용합니다.

**정상:** `title`·`note` 의 before/after 와 코드포인트 길이가 출력됨

**의미:** NFC 정규화로 분해된 한글(NFD)을 완성형으로 합치고, 잘린 이모지가 남긴 외톨이 surrogate 를 안전한 대체 문자(U+FFFD)로 치환합니다(보이지 않는 zero-width 제거는 이 함수의 범위가 아닙니다).

In [ ]:
import unicodedata

# (en) Realistic: macOS often pastes Hangul as NFD (decomposed jamo); a truncated emoji leaves a lone surrogate.
# (kr) 현실 예: macOS는 한글을 NFD(자모 분해)로 붙여넣는 경우가 많고, 잘린 이모지는 외톨이 surrogate를 남긴다.
sample = {"title": unicodedata.normalize("NFD", "회의록"), "note": "보고서\ud83d"}
norm = exaone.output.normalize_json_string_values(sample)
print("title before:", repr(sample["title"]), "| code points:", len(sample["title"]))
print("title after :", repr(norm["title"]), "| code points:", len(norm["title"]))
print("note  before:", repr(sample["note"]), "| code points:", len(sample["note"]))
print("note  after :", repr(norm["note"]), "| code points:", len(norm["note"]))

**출력 해석:** `normalize_json_string_values` 는 **NFC 정규화 + 외톨이 surrogate 대체**를 합니다 — 출력에서 직접 확인됩니다.

- `title`: 눈에는 같은 `회의록` 이지만 코드포인트가 **7 → 3** 으로 줄었습니다. macOS 가 붙여넣은 NFD(자모 분해) 한글이 NFC(완성형)로 합쳐진 것입니다.
- `note`: 잘린 이모지가 남긴 외톨이 surrogate(`\ud83d`)가 U+FFFD(대체 문자)로 치환됩니다 — 원본 이모지를 되살리는 게 아니라 깨진 바이트를 무력화하는 것이라, 코드포인트가 **4 → 6** 으로 늘어납니다.
- **한계:** 이 함수는 zero-width(`\u200b`)나 full-width 같은 호환 문자는 건드리지 않습니다(NFKC 가 아님). zero-width 류는 Session 2-1 의 reference-text sanitizer 영역입니다.

### Session 2-3. 산출물 — `injection_defense.json`

**하는 일:** injection 방어 결과를 `injection_defense.json` 으로 저장합니다.

**정상:** `saved:` 경로가 출력됨

**의미:** 케이스별 `vector`·`pass`·`changed`·`preview` 가 남아 다음 Session 이나 회귀 테스트의 입력이 됩니다.

In [ ]:
report = {"generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "pass_count": passed, "total": len(rows), "cases": rows}
path = out_dir / "injection_defense.json"
path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", path.resolve())


**출력 해석:** `saved:` 가 보이면 `injection_defense.json` 이 기록된 것입니다.

- 케이스별 `vector`·`pass`·`changed`·`preview` 가 함께 저장돼, 어떤 입력이 어떻게 sanitize 됐는지 사후에 추적할 수 있습니다.
- 회귀 테스트가 이 파일을 읽어 "10건 방어"가 유지되는지 검증합니다.

## Session 3. Observability Fields & Traces

"trace 하지 않으면 디버깅도 평가도 못 합니다." 표준 필드 상수(exaone.observability.fields)를 써서 모든 팀·도구가 같은 키 로 로그를 남기게 합니다.


### Session 3-1. 필드 glossary (상수)

**하는 일:** observability 필드 glossary 상수를 출력합니다.

**정상:** `{label:20} -> {const}` 가 보임

**의미:** 문자열을 직접 쓰지 말고 상수를 참조하면 오타·불일치가 사라집니다.


In [ ]:
obs_fields = exaone.observability.fields
GLOSSARY = [
("request_id", obs_fields.REQUEST_ID),
("model", obs_fields.MODEL),
("latency_ms", obs_fields.LATENCY_MS),
("agent", obs_fields.AGENT),
("turns_used", obs_fields.TURNS_USED),
("tool_invocations", obs_fields.TOOL_INVOCATIONS),
("llm_calls", obs_fields.LLM_CALLS),
("llm.empty_content", obs_fields.LLM_EMPTY_CONTENT),
]
for label, const in GLOSSARY:
    print(f"{label:20} -> {const}")


**출력 해석:** 8개 관측 필드 상수가 `이름 -> 값` 으로 출력됩니다.

- `request_id`·`latency_ms`·`turns_used`·`tool_invocations`·`llm_calls` 등은 문자열을 직접 쓰지 않고 `exaone.observability.fields` 의 상수를 참조합니다.
- 모든 팀·도구가 같은 키로 로그를 남기면 오타·불일치가 사라지고, 트레이스를 기계적으로 집계·평가할 수 있습니다.

### Session 3-2. 짧은 `ToolAgent` run → trace 이벤트

**하는 일:** 키가 있으면 `ping` 도구를 단 `ToolAgent` 를 실행하고 `run_end` 트레이스를 표준 관측 필드로 출력합니다(키 없으면 `run_start` 만 기록하고 `[SKIP]`).

**정상:** `events: 2` 와 `turns_used`·`tool_invocations`·`llm_calls` 가 담긴 `run_end` JSON

**의미:** 실제 실행 메타(turns_used·tool_invocations·llm_calls)를 표준 필드로 trace 에 남깁니다.

In [ ]:
REQUEST_ID = str(uuid.uuid4())
events = [{"ts": datetime.now(timezone.utc).isoformat(), "event": "run_start", "request_id": REQUEST_ID}]

if HAS_API:
    reg = exaone.tools.ToolRegistry()
    reg.register(exaone.tools.tool_from_callable(
        "ping",
        {"type": "function", "function": {"name": "ping", "description": "Return pong",
         "parameters": {"type": "object", "properties": {}, "additionalProperties": False}}},
        lambda _n, _a: exaone.tools.ToolResult.success(content="pong", source="ping").to_dict(),
    ))
    agent = exaone.agents.ToolAgent(tool_registry=reg, use_thinking_router=False, max_turns=3)
    t0 = time.monotonic()
    res = agent.run(exaone.agents.AgentContext(query="ping 도구를 호출한 뒤 결과를 한국어 존댓말로 알려주세요."), llm=client)
    meta = res.metadata or {}
    events.append({
        "event": "run_end", "request_id": REQUEST_ID, "success": res.success,
        "latency_ms": round((time.monotonic() - t0) * 1000, 1), "model": client.model,
        "turns_used": meta.get("turns_used"), "tool_invocations": meta.get("tool_invocations"),
        "llm_calls": meta.get("llm_calls"), "answer_preview": (res.content or "")[:120],
    })
    print("events:", len(events))
    # (en) Surface the run_end trace so the standard observability fields are visible inline.
    # (kr) run_end 트레이스를 출력해 표준 관측 필드를 셀 안에서 바로 보이게 한다.
    print(json.dumps(events[-1], ensure_ascii=False, indent=2))
else:
    # (en) No key: keep only run_start so Session 3-3 can still write the trace offline.
    # (kr) 키 없음: run_start 만 남겨 Session 3-3 가 오프라인에서도 트레이스를 기록.
    print("[SKIP] LLM 키 없음 — run_end 트레이스는 키 필요. events:", len(events))

**출력 해석:** `events: 2` 와 함께 `run_end` 트레이스가 표준 필드로 출력됩니다(키 없으면 `run_start` 만).

- `success`, `latency_ms`, `model`, `llm_calls`(각 내부 단계의 지연을 단계명과 함께; 도구를 호출하면 `enrich_react` 같은 단계가 추가됩니다)가 한 이벤트에 담깁니다 — 이것이 디버깅·평가의 단위입니다.
- `tool_invocations` 는 모델이 실제로 `ping` 을 호출했는지를 그대로 보여줍니다(샘플링에 따라 0 이 될 수도 있음). 트레이스는 "도구를 썼는지/안 썼는지"까지 사실대로 남깁니다.

### Session 3-3. 산출물 — `session_trace.jsonl` + `field_glossary.md`

**하는 일:** 앞 단계의 `events` 를 `session_trace.jsonl` 로, 필드 상수를 `field_glossary.md` 로 저장합니다.

**정상:** 두 개의 `saved:` 경로(session_trace.jsonl·field_glossary.md)가 출력됨

**의미:** `field_glossary.md` 는 LLM 응답이 아닌 상수 목록에서 만들어져 라이브 호출 없이도 생성됩니다 — 팀 wiki 에 붙일 표준 필드 사전입니다.

In [ ]:
jsonl_path = out_dir / "session_trace.jsonl"
with jsonl_path.open("w", encoding="utf-8") as fh:
    for ev in events:
        fh.write(json.dumps(ev, ensure_ascii=False) + "\n")

glossary_md = "\n".join(
["# Observability field glossary (Track 07)", "", "| Concept | Constant |", "|---|---|"]
+ [f"| {label} | `{const}` |" for label, const in GLOSSARY]
+ ["", "See `exaone/observability/fields.py` for the full list.", ""]
)
(out_dir / "field_glossary.md").write_text(glossary_md, encoding="utf-8")
print("saved:", jsonl_path.resolve())
print("saved:", (out_dir / "field_glossary.md").resolve())


**출력 해석:** 두 개의 `saved:` 경로가 `session_trace.jsonl` 과 `field_glossary.md` 입니다.

- `session_trace.jsonl` 은 한 줄당 한 이벤트(JSONL)라 스트리밍 로그처럼 append 할 수 있습니다(키 없으면 `run_start` 한 줄).
- `field_glossary.md` 는 LLM 응답이 아닌 상수 목록(GLOSSARY)에서 생성돼 라이브 호출 결과에 의존하지 않습니다 — 팀 wiki 에 붙일 표준 필드 사전입니다.

## Wrap-up. 마무리

이 노트북에서는 에이전트 운영 리스크를 세 겹으로 줄였습니다.

- **Session 1 — HITL:** 승인 게이트를 `sensitive_publish.execute()` 안에 두고 승인/거부/읽기전용 3건을 검증했습니다. 거부는 `ok=False, error="rejected_by_human"` 로 남았고(`outcome` 은 일반 실패 버킷 `transport_error` 라 `error` 로 판정), 키가 있으면 라이브 `ToolAgent` 가 `safe_lookup` 근거로 높임말 구조화 답변을 만들었습니다.
- **Session 2 — Injection 방어:** injection 10건에서 가짜 태그는 `[removed-tag:...]`, 비밀은 `[REDACTED]` 로 무력화했습니다. 단, 자연어 지시("Ignore previous instructions…")는 reference 데이터로 그대로 남습니다 — sanitizer 는 *구조와 비밀* 을 막지 *의미* 를 지우지 않습니다. NFC 정규화는 분해 한글을 합치고 외톨이 surrogate 를 U+FFFD 로 치환하지만 zero-width 는 범위 밖입니다.
- **Session 3 — 관측:** 표준 필드 상수로 `ToolAgent` 실행을 `run_end` 트레이스(JSONL)와 필드 glossary 로 남겼습니다.

**핵심 takeaways**

- 승인 로직은 프롬프트가 아니라 **도구 실행 안**에 둬야 LLM 우회를 막습니다.
- 인젝션 방어는 "다 막았다"가 아니라 **무엇을 막고 무엇이 남는지**(구조·비밀 vs 자연어 데이터)를 아는 것이 중요합니다.
- 관측은 **표준 필드 상수**로 통일해야 트레이스를 기계적으로 집계·평가할 수 있습니다.

**한계 / 주의**

- 라이브 셀(Session 1-3·3-2)은 키가 없으면 `[SKIP]` 되고, 나머지 오프라인 로직은 키 없이 실행됩니다.
- `ToolResult.failure()` 의 `outcome=transport_error` 는 거부의 정확한 의미가 아니므로 `error` 필드로 판정합니다.
- `normalize_json_string_values` 는 NFC + surrogate 대체만 하며 zero-width 제거는 하지 않습니다.

## 체크포인트

- [ ] Session 1 `hitl_trace.json` — 거부 시나리오가 `rejected_by_human` 으로 실패 (assert 통과).
- [ ] Session 2 `injection_defense.json` — injection 10건 **모두 pass** (키 없이).
- [ ] Session 3 `field_glossary.md` — 표준 필드 상수가 채워진다 (키 없이).

**다음:** Track 08 — Evaluation (또는 심화 `07b_slo_production_defaults`)